# Oracle Simulation Results Analysis

Compares speculative decoding methods across workloads:
- W1: BFCLv4 web_search (req0-12)
- W2: BFCLv4 memory (req0-15)
- W3: BFCLv3 multi-turn (req0-20)
- W4: SpecBench (req0-20)

Methods: EU Oracle, Choose-One, single proposers, hybrids, EU pairwise

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

matplotlib.rcParams.update({
    'font.size': 11,
    'figure.figsize': (14, 5),
    'figure.dpi': 100,
})

In [ ]:
# Load all workload results
BASE = "../results/qwen3_8b"
WORKLOADS = {
    "W1: BFCLv4 web_search": f"{BASE}/bfcl_v4_req0-12/tree_oracle_sim.json",
    "W2: BFCLv4 memory":     f"{BASE}/bfcl_v4_req0-15/tree_oracle_sim.json",
    "W3: BFCLv3 multi-turn": f"{BASE}/bfcl_v3_req0-20/tree_oracle_sim.json",
    "W4: SpecBench":         f"{BASE}/specbench_req0-20/tree_oracle_sim.json",
}

all_data = {}
for name, path in WORKLOADS.items():
    try:
        with open(path) as f:
            all_data[name] = json.load(f)
        print(f"✓ {name}: loaded")
    except Exception as e:
        print(f"✗ {name}: {e}")

In [ ]:
def extract_budget_table(data):
    """Extract per-budget results into a DataFrame."""
    if "budgets" in data:
        # New format: budgets dict
        rows = []
        for b_str, r in sorted(data["budgets"].items(), key=lambda x: int(x[0])):
            row = {"budget": int(b_str)}
            row.update(r)
            rows.append(row)
        return pd.DataFrame(rows)
    elif "latency" in data:
        # Old format: latency.budget_sweep
        return pd.DataFrame(data["latency"]["budget_sweep"])
    return pd.DataFrame()

tables = {}
for name, data in all_data.items():
    tables[name] = extract_budget_table(data)
    print(f"{name}: {len(tables[name])} budgets, columns: {list(tables[name].columns)[:10]}...")

## 1. Best Speedup per Method (All Workloads)

In [ ]:
def get_best_speedups(df):
    """Find best budget and speedup for each method."""
    results = {}
    for col in df.columns:
        if col.endswith('_speedup'):
            method = col.replace('_speedup', '')
            idx = df[col].idxmax()
            results[method] = {
                'speedup': df[col].iloc[idx],
                'budget': df['budget'].iloc[idx],
                'mat': df.get(f'{method}_mat', pd.Series([0])).iloc[idx] if f'{method}_mat' in df.columns else 0,
            }
    return results

# Build comparison table
all_methods = set()
best_per_workload = {}
for name, df in tables.items():
    best = get_best_speedups(df)
    best_per_workload[name] = best
    all_methods.update(best.keys())

# Create summary DataFrame
method_order = ['eu', 'c1', 'eagle3', 'suffix', 'draft_model',
                'eagle3+suffix', 'eagle3+draft_model', 'draft_model+suffix',
                'eu_eagle3+suffix', 'eu_eagle3+draft_model', 'eu_draft_model+suffix',
                'hybrid_e3_t1.0', 'hybrid_e3_t3.0', 'hybrid_e3_t5.0',
                'hybrid_dm_t1.0', 'hybrid_dm_t3.0', 'hybrid_dm_t5.0']
methods_present = [m for m in method_order if m in all_methods]
# Add any remaining methods not in order
methods_present += sorted(all_methods - set(methods_present))

rows = []
for method in methods_present:
    row = {'method': method}
    for wname in tables:
        b = best_per_workload[wname].get(method)
        if b:
            row[f'{wname}'] = f"{b['speedup']:.2f}x (B={b['budget']})"
        else:
            row[f'{wname}'] = '-'
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('method')
summary_df

## 2. Per-Workload Detailed Tables

In [ ]:
for name, df in tables.items():
    print(f"\n{'='*80}")
    print(f"{name}")
    print(f"{'='*80}")
    
    # Select speedup columns
    speedup_cols = [c for c in df.columns if c.endswith('_speedup')]
    display_cols = ['budget']
    if 'target_forward_ms' in df.columns:
        display_cols.append('target_forward_ms')
    if 'eagle3_draft_ms' in df.columns:
        display_cols.append('eagle3_draft_ms')
    display_cols += speedup_cols
    
    sub = df[display_cols].copy()
    # Rename columns for readability
    sub.columns = [c.replace('_speedup', '') for c in sub.columns]
    
    print(sub.to_string(index=False, float_format='{:.2f}'.format))
    
    # Best per method
    best = get_best_speedups(df)
    print(f"\nBest:")
    for method, b in sorted(best.items(), key=lambda x: -x[1]['speedup']):
        print(f"  {method:>25}: {b['speedup']:.2f}x (B={b['budget']})")

## 3. Speedup vs Budget (All Workloads)

In [ ]:
# Key methods to compare
key_methods = ['eu_speedup', 'c1_speedup', 'eagle3_speedup', 'suffix_speedup']
method_labels = {'eu_speedup': 'EU Oracle', 'c1_speedup': 'Choose-One',
                 'eagle3_speedup': 'EAGLE3', 'suffix_speedup': 'Suffix'}
method_colors = {'eu_speedup': '#d62728', 'c1_speedup': '#1f77b4',
                 'eagle3_speedup': '#2ca02c', 'suffix_speedup': '#ff7f0e'}

n_workloads = len(tables)
fig, axes = plt.subplots(1, n_workloads, figsize=(5 * n_workloads, 5), sharey=True)
if n_workloads == 1:
    axes = [axes]

for ax, (name, df) in zip(axes, tables.items()):
    for method in key_methods:
        if method in df.columns:
            ax.plot(df['budget'], df[method], 'o-',
                    label=method_labels.get(method, method),
                    color=method_colors.get(method), linewidth=2, markersize=5)
    ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3)
    ax.set_xlabel('Budget')
    ax.set_title(name.split(':')[0] + ':\n' + name.split(': ')[1], fontsize=11)
    ax.set_xscale('log', base=2)
    ax.grid(alpha=0.2)

axes[0].set_ylabel('Speedup vs Vanilla')
axes[-1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.suptitle('Core Methods: Speedup vs Budget', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Pairwise & Hybrid Methods

In [ ]:
pair_methods = {
    'eagle3+suffix_speedup': 'C1(E3+Sfx)',
    'eu_eagle3+suffix_speedup': 'EU(E3+Sfx)',
    'draft_model+suffix_speedup': 'C1(DM+Sfx)',
    'eu_draft_model+suffix_speedup': 'EU(DM+Sfx)',
    'eagle3+draft_model_speedup': 'C1(E3+DM)',
    'eu_eagle3+draft_model_speedup': 'EU(E3+DM)',
}

n_workloads = len(tables)
fig, axes = plt.subplots(1, n_workloads, figsize=(5 * n_workloads, 5), sharey=True)
if n_workloads == 1:
    axes = [axes]

for ax, (name, df) in zip(axes, tables.items()):
    for method, label in pair_methods.items():
        if method in df.columns:
            ax.plot(df['budget'], df[method], 'o-', label=label,
                    linewidth=2, markersize=5)
    ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3)
    ax.set_xlabel('Budget')
    ax.set_title(name.split(':')[0], fontsize=11)
    ax.set_xscale('log', base=2)
    ax.grid(alpha=0.2)

axes[0].set_ylabel('Speedup')
axes[-1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.suptitle('Pairwise Methods: Choose-One vs EU Oracle', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Hybrid methods
hybrid_methods = {
    'hybrid_e3_t1.0_speedup': 'Hybrid(Sfx→E3) t=1',
    'hybrid_e3_t3.0_speedup': 'Hybrid(Sfx→E3) t=3',
    'hybrid_e3_t5.0_speedup': 'Hybrid(Sfx→E3) t=5',
    'hybrid_dm_t1.0_speedup': 'Hybrid(Sfx→DM) t=1',
    'hybrid_dm_t3.0_speedup': 'Hybrid(Sfx→DM) t=3',
    'hybrid_dm_t5.0_speedup': 'Hybrid(Sfx→DM) t=5',
}

fig, axes = plt.subplots(1, n_workloads, figsize=(5 * n_workloads, 5), sharey=True)
if n_workloads == 1:
    axes = [axes]

for ax, (name, df) in zip(axes, tables.items()):
    for method, label in hybrid_methods.items():
        if method in df.columns:
            ax.plot(df['budget'], df[method], 'o-', label=label,
                    linewidth=2, markersize=5)
    # Add suffix baseline
    if 'suffix_speedup' in df.columns:
        ax.plot(df['budget'], df['suffix_speedup'], 's--', label='Suffix only',
                color='gray', linewidth=1, markersize=4)
    ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3)
    ax.set_xlabel('Budget')
    ax.set_title(name.split(':')[0], fontsize=11)
    ax.set_xscale('log', base=2)
    ax.grid(alpha=0.2)

axes[0].set_ylabel('Speedup')
axes[-1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.suptitle('Hybrid Methods: Suffix → Fallback', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Latency Decomposition

In [ ]:
# Use first workload's latency data (same latency config for all)
first_df = list(tables.values())[0]
if 'target_forward_ms' in first_df.columns and 'eagle3_draft_ms' in first_df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    
    budgets = first_df['budget']
    t_fwd = first_df['target_forward_ms']
    e3_draft = first_df['eagle3_draft_ms']
    
    # Read vanilla from latency config
    try:
        lc = json.load(open(f"{BASE}/bfcl_v4_req0-12/latency_config.json"))
        vanilla = lc['vanilla_step_ms']
        dm_tpot = lc.get('draft_lm_tpot_ms', 0)
    except:
        vanilla = 21.07
        dm_tpot = 16.19
    
    # Stacked bar: target_forward + eagle3_draft + overhead
    eagle3_step = [lc.get('eagle3_step_ms', {}).get(str(b), t + e) 
                   for b, t, e in zip(budgets, t_fwd, e3_draft)]
    overhead = [s - t - e for s, t, e in zip(eagle3_step, t_fwd, e3_draft)]
    
    x = np.arange(len(budgets))
    w = 0.5
    
    ax.bar(x, t_fwd, w, label='Target Forward', color='#1f77b4')
    ax.bar(x, e3_draft, w, bottom=t_fwd, label='EAGLE3 Draft', color='#ff7f0e')
    ax.bar(x, overhead, w, bottom=[t+e for t,e in zip(t_fwd, e3_draft)],
           label='Overhead', color='#d62728', alpha=0.5)
    
    ax.axhline(y=vanilla, color='gray', linestyle='--', label=f'Vanilla ({vanilla:.1f}ms)')
    
    # Draft model cost line
    if dm_tpot > 0:
        dm_costs = [b * dm_tpot for b in budgets]
        ax.plot(x, dm_costs, 'gD--', label=f'Draft Model (B×{dm_tpot:.1f}ms)', markersize=5)
    
    ax.set_xticks(x)
    ax.set_xticklabels([str(b) for b in budgets])
    ax.set_xlabel('Budget')
    ax.set_ylabel('Latency (ms)')
    ax.set_title('Step Cost Decomposition per Budget')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    # Table
    lat_df = pd.DataFrame({
        'Budget': budgets,
        'T_fwd (ms)': t_fwd,
        'E3_draft (ms)': e3_draft,
        'Step (ms)': eagle3_step,
        'Overhead (ms)': overhead,
        'DM cost (ms)': [b * dm_tpot for b in budgets],
    })
    print(lat_df.to_string(index=False, float_format='{:.1f}'.format))
else:
    print('No decomposed latency data available')

## 6. Cross-Workload Comparison (Best Method per Workload)

In [ ]:
# Heatmap: best speedup per method per workload
heat_data = []
for method in methods_present:
    row = []
    for wname in tables:
        b = best_per_workload[wname].get(method)
        row.append(b['speedup'] if b else 0)
    heat_data.append(row)

heat_df = pd.DataFrame(heat_data, index=methods_present,
                        columns=[n.split(': ')[1] for n in tables])

fig, ax = plt.subplots(figsize=(10, max(6, len(methods_present) * 0.4)))
im = ax.imshow(heat_df.values, cmap='YlOrRd', aspect='auto')

ax.set_xticks(range(len(heat_df.columns)))
ax.set_xticklabels(heat_df.columns, fontsize=10)
ax.set_yticks(range(len(heat_df.index)))
ax.set_yticklabels(heat_df.index, fontsize=9)

# Annotate cells
for i in range(len(heat_df.index)):
    for j in range(len(heat_df.columns)):
        v = heat_df.values[i, j]
        if v > 0:
            color = 'white' if v > heat_df.values.max() * 0.6 else 'black'
            ax.text(j, i, f'{v:.2f}x', ha='center', va='center',
                    fontsize=8, color=color)

plt.colorbar(im, label='Speedup')
ax.set_title('Best Speedup: Method × Workload', fontsize=13)
plt.tight_layout()
plt.show()

## 7. EU Oracle vs Choose-One Gap

In [ ]:
fig, axes = plt.subplots(1, n_workloads, figsize=(5 * n_workloads, 5), sharey=True)
if n_workloads == 1:
    axes = [axes]

for ax, (name, df) in zip(axes, tables.items()):
    if 'eu_speedup' in df.columns and 'c1_speedup' in df.columns:
        gap_pct = (df['eu_speedup'] / df['c1_speedup'] - 1) * 100
        colors = ['#2ca02c' if g >= 0 else '#d62728' for g in gap_pct]
        ax.bar(range(len(df)), gap_pct, color=colors)
        ax.set_xticks(range(len(df)))
        ax.set_xticklabels(df['budget'])
        ax.axhline(y=0, color='black', linewidth=0.5)
        ax.set_xlabel('Budget')
        ax.set_title(name.split(':')[0], fontsize=11)
        ax.grid(alpha=0.2, axis='y')

axes[0].set_ylabel('EU advantage over Choose-One (%)')
plt.suptitle('EU Oracle vs Choose-One: When does DP help?', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Summary Table

In [ ]:
print("=" * 90)
print("ORACLE SIMULATION SUMMARY — Qwen3-8B")
print("=" * 90)

for name in tables:
    best = best_per_workload[name]
    best_method = max(best.items(), key=lambda x: x[1]['speedup'])
    eu = best.get('eu', {'speedup': 0, 'budget': 0})
    c1 = best.get('c1', {'speedup': 0, 'budget': 0})
    
    print(f"\n{name}")
    print(f"  Best overall: {best_method[0]} → {best_method[1]['speedup']:.2f}x (B={best_method[1]['budget']})")
    print(f"  EU Oracle:    {eu['speedup']:.2f}x (B={eu['budget']})")
    print(f"  Choose-One:   {c1['speedup']:.2f}x (B={c1['budget']})")
    if c1['speedup'] > 0:
        print(f"  EU advantage: {(eu['speedup']/c1['speedup']-1)*100:+.1f}%")

print("\n" + "=" * 90)